<a href="https://colab.research.google.com/github/profvivi/testepython/blob/main/testepython.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import base64
import hashlib
import json
import os
from google.colab import output
from IPython.display import HTML, JSON, display

ARQUIVO_COFRE = "cofre_saude_emergencia.txt"


# --- 1. LÓGICA DE CRIPTOGRAFIA (PYTHON / BACK-END) ---
def gerar_chave(senha_mestra):
  return hashlib.sha256(senha_mestra.encode("utf-8")).digest()


def criptografar(texto, senha_mestra):
  chave = gerar_chave(senha_mestra)
  texto_bytes = texto.encode("utf-8")
  resultado = bytearray()
  for i in range(len(texto_bytes)):
    resultado.append(texto_bytes[i] ^ chave[i % len(chave)])
  return base64.b64encode(resultado).decode("utf-8")


def descriptografar(texto_criptografado, senha_mestra):
  try:
    chave = gerar_chave(senha_mestra)
    dados_bytes = base64.b64decode(texto_criptografado.encode("utf-8"))
    resultado = bytearray()
    for i in range(len(dados_bytes)):
      resultado.append(dados_bytes[i] ^ chave[i % len(chave)])
    return resultado.decode("utf-8")
  except Exception:
    return None


def listar_titulos():
  titulos = []
  if os.path.exists(ARQUIVO_COFRE):
    with open(ARQUIVO_COFRE, "r", encoding="utf-8") as f:
      for linha in f:
        if " | " in linha:
          titulos.append(linha.split(" | ")[0])
  return titulos


# --- 2. PONTE DE COMUNICAÇÃO ENTRE JAVASCRIPT E PYTHON ---
def salvar_no_python(titulo, segredo, senha):
  if not titulo or not segredo or not senha:
    return JSON({"status": "erro", "msg": "Por favor, preencha todos os 3 campos!"})

  dado_protegido = criptografar(segredo, senha)
  with open(ARQUIVO_COFRE, "a", encoding="utf-8") as f:
    f.write(f"{titulo} | {dado_protegido}\n")

  return JSON({
      "status": "ok",
      "msg": f"'{titulo}' foi trancado com segurança no cofre médico!",
      "itens": listar_titulos(),
  })


def abrir_no_python(indice, senha):
  if not os.path.exists(ARQUIVO_COFRE):
    return JSON({"status": "erro", "msg": "O cofre médico ainda está vazio."})

  with open(ARQUIVO_COFRE, "r", encoding="utf-8") as f:
    linhas = f.readlines()

  if indice < 0 or indice >= len(linhas):
    return JSON({"status": "erro", "msg": "Selecione um registro válido."})

  titulo, dado_cripto = linhas[indice].strip().split(" | ")
  revelado = descriptografar(dado_cripto, senha)

  if revelado is not None:
    return JSON({"status": "ok", "titulo": titulo, "dado": revelado})
  else:
    return JSON({"status": "erro", "msg": "Senha Mestra incorreta! Acesso bloqueado."})


output.register_callback("notebook.salvar_saude", salvar_no_python)
output.register_callback("notebook.abrir_saude", abrir_no_python)

itens_iniciais = json.dumps(listar_titulos())

# --- 3. INTERFACE VISUAL ACESSÍVEL (HTML + CSS + JAVASCRIPT) ---
html_code = f"""
<div style="font-family: Arial, sans-serif; max-width: 560px; margin: 10px auto; background: #ffffff; border-radius: 14px; box-shadow: 0 6px 18px rgba(0,0,0,0.12); overflow: hidden; border: 2px solid #cbd5e1;">

  <!-- Cabeçalho Temático -->
  <div style="background: #991b1b; color: white; padding: 20px; text-align: center;">
    <h2 style="margin: 0; font-size: 24px;">🏥 COFRE DE SAÚDE E EMERGÊNCIA</h2>
    <p style="margin: 6px 0 0; font-size: 14px; color: #fecaca;">Contatos médicos, alergias e dados vitais protegidos</p>
  </div>

  <!-- Aviso Educativo Contra Golpes -->
  <div style="background: #fef3c7; color: #92400e; padding: 12px 18px; font-size: 14px; border-bottom: 1px solid #fde68a; line-height: 1.4;">
    ⚠️ <b>Dica de Segurança:</b> Hospitais e planos de saúde <b>nunca ligam pedindo senhas ou códigos por WhatsApp</b>. Compartilhe sua Senha Mestra apenas com familiares de confiança.
  </div>

  <div style="padding: 22px;">
    <h3 style="margin-top: 0; color: #1e293b; font-size: 18px;">1. GUARDAR INFORMAÇÃO MÉDICA</h3>

    <label style="display:block; font-weight:bold; font-size:15px; margin-bottom:5px; color:#334155;">O que você quer guardar?</label>
    <input id="titulo" type="text" placeholder="Ex: Contato Dr. Carlos, Tipo Sanguíneo, Alergias" style="width:100%; padding:12px; margin-bottom:14px; border:2px solid #94a3b8; border-radius:8px; font-size:16px; box-sizing:border-box;">

    <label style="display:block; font-weight:bold; font-size:15px; margin-bottom:5px; color:#334155;">Informação Secreta (telefone, remédio, carteirinha):</label>
    <input id="segredo" type="text" placeholder="Digite os detalhes aqui" style="width:100%; padding:12px; margin-bottom:14px; border:2px solid #94a3b8; border-radius:8px; font-size:16px; box-sizing:border-box;">

    <label style="display:block; font-weight:bold; font-size:15px; margin-bottom:5px; color:#334155;">Crie uma Senha Mestra para trancar:</label>
    <input id="senhaCad" type="password" placeholder="Digite sua senha secreta" style="width:100%; padding:12px; margin-bottom:18px; border:2px solid #94a3b8; border-radius:8px; font-size:16px; box-sizing:border-box;">

    <button onclick="acaoSalvar()" style="width:100%; padding:14px; background:#15803d; color:white; border:none; border-radius:8px; font-size:18px; font-weight:bold; cursor:pointer;">
      🔒 TRANCAR DADO MÉDICO
    </button>

    <hr style="margin: 26px 0; border: 0; border-top: 2px solid #e2e8f0;">

    <h3 style="margin-top: 0; color: #1e293b; font-size: 18px;">2. CONSULTAR EM CASO DE EMERGÊNCIA</h3>

    <label style="display:block; font-weight:bold; font-size:15px; margin-bottom:5px; color:#334155;">Escolha o registro guardado:</label>
    <select id="listaItens" style="width:100%; padding:12px; margin-bottom:14px; border:2px solid #94a3b8; border-radius:8px; font-size:16px; background:white;">
    </select>

    <label style="display:block; font-weight:bold; font-size:15px; margin-bottom:5px; color:#334155;">Digite a Senha Mestra:</label>
    <input id="senhaAbrir" type="password" placeholder="Senha para destrancar" style="width:100%; padding:12px; margin-bottom:18px; border:2px solid #94a3b8; border-radius:8px; font-size:16px; box-sizing:border-box;">

    <button onclick="acaoAbrir()" style="width:100%; padding:14px; background:#1d4ed8; color:white; border:none; border-radius:8px; font-size:18px; font-weight:bold; cursor:pointer;">
      🔓 DESTRANCAR INFORMAÇÃO
    </button>

    <div id="caixaResultado" style="display:none; margin-top:20px; padding:16px; border-radius:8px; font-size:16px; line-height:1.5;"></div>
  </div>
</div>

<script>
  function atualizarSelect(itens) {{
    const select = document.getElementById('listaItens');
    select.innerHTML = '';
    if (itens.length === 0) {{
      select.innerHTML = '<option value="-1">Nenhum dado médico guardado ainda</option>';
      return;
    }}
    itens.forEach((item, index) => {{
      const opt = document.createElement('option');
      opt.value = index;
      opt.textContent = '🔒 ' + item;
      select.appendChild(opt);
    }});
  }}

  function mostrarMensagem(texto, tipo) {{
    const caixa = document.getElementById('caixaResultado');
    caixa.style.display = 'block';
    caixa.innerHTML = texto;
    if (tipo === 'ok') {{
      caixa.style.background = '#dcfce7';
      caixa.style.color = '#14532d';
      caixa.style.border = '2px solid #86efac';
    }} else {{
      caixa.style.background = '#fee2e2';
      caixa.style.color = '#7f1d1d';
      caixa.style.border = '2px solid #fca5a5';
    }}
  }}

  async function acaoSalvar() {{
    const titulo = document.getElementById('titulo').value.trim();
    const segredo = document.getElementById('segredo').value.trim();
    const senha = document.getElementById('senhaCad').value.trim();

    const resposta = await google.colab.kernel.invokeFunction('notebook.salvar_saude', [titulo, segredo, senha], {{}});
    const dados = resposta.data['application/json'];

    if (dados.status === 'ok') {{
      document.getElementById('titulo').value = '';
      document.getElementById('segredo').value = '';
      document.getElementById('senhaCad').value = '';
      atualizarSelect(dados.itens);
      mostrarMensagem('✅ <b>Sucesso:</b> ' + dados.msg, 'ok');
    }} else {{
      mostrarMensagem('⚠️ <b>Atenção:</b> ' + dados.msg, 'erro');
    }}
  }}

  async function acaoAbrir() {{
    const indice = parseInt(document.getElementById('listaItens').value);
    const senha = document.getElementById('senhaAbrir').value.trim();

    const resposta = await google.colab.kernel.invokeFunction('notebook.abrir_saude', [indice, senha], {{}});
    const dados = resposta.data['application/json'];
    document.getElementById('senhaAbrir').value = '';

    if (dados.status === 'ok') {{
      mostrarMensagem('🏥 <b>' + dados.titulo + ':</b><br><span style="font-size:20px; font-weight:bold;">' + dados.dado + '</span>', 'ok');
    }} else {{
      mostrarMensagem('❌ <b>Acesso Negado:</b> ' + dados.msg, 'erro');
    }}
  }}

  atualizarSelect({itens_iniciais});
</script>
"""

display(HTML(html_code))